# Orbit Wars — RL Development Notebook

Development notebook for the [Kaggle Orbit Wars competition](https://www.kaggle.com/competitions/orbit-wars).

**Goal:** learn reinforcement learning by building competitive agents.

| Section | Purpose |
|---------|---------|
| 1. Setup | Install deps, detect runtime (Kaggle / Colab / local) |
| 2. Baseline agent | Deterministic rule-based benchmark |
| 3. RL training | PPO agent via Stable-Baselines3 |
| 4. Evaluation | Baseline vs RL head-to-head |
| 5. Submission | Write `submission.py` for Kaggle |

## 1. Setup

In [ ]:
import os, sys

def detect_env():
    if os.path.exists('/kaggle/input'):
        return 'kaggle'
    try:
        import google.colab          # noqa: F401
        return 'colab'
    except ImportError:
        pass
    return 'local'

ENV = detect_env()
print(f"Runtime: {ENV}")

In [ ]:
# Install / upgrade dependencies based on runtime
if ENV in ('kaggle', 'colab'):
    os.system('pip install -q --upgrade "kaggle-environments>=1.28.0"')

if ENV == 'colab':
    os.system('pip install -q "stable-baselines3[extra]>=2.0" gymnasium torch')
elif ENV == 'local':
    # Assume a requirements.txt or manual install; just verify imports below.
    pass

In [ ]:
# Clone / mount project files so imports work in Colab / Kaggle
if ENV == 'colab':
    # Option A: clone from GitHub (replace with your repo URL)
    REPO_URL = "https://github.com/YOUR_USERNAME/orbit_wars.git"
    if not os.path.exists('orbit_wars'):
        os.system(f'git clone {REPO_URL} orbit_wars')
    os.chdir('orbit_wars')
    print("Working directory:", os.getcwd())
elif ENV == 'kaggle':
    # Files uploaded as a dataset will be in /kaggle/input/orbit-wars-code/
    # Adjust path if you named your dataset differently.
    DATASET_PATH = '/kaggle/input/orbit-wars-code'
    if os.path.exists(DATASET_PATH) and DATASET_PATH not in sys.path:
        sys.path.insert(0, DATASET_PATH)
        print(f"Added {DATASET_PATH} to sys.path")
    else:
        print("Dataset not found — using inline definitions below as fallback.")
# local: nothing needed

In [ ]:
import math, time, warnings
import numpy as np
warnings.filterwarnings('ignore')

import torch
print(f"PyTorch {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"  GPU: {torch.cuda.get_device_name(0)}")

from kaggle_environments import make
print("kaggle-environments imported OK")

## 2. Deterministic Baseline Agent

Rule-based agent: for each owned planet, targets the weakest nearby planet,
accounts for travel time, avoids the sun, and prevents duplicate targeting.
This is our benchmark — every RL agent must beat it.

In [ ]:
try:
    from agents.baseline import agent as baseline_agent
    print("Imported baseline from agents/baseline.py")
except ImportError:
    # Fallback: inline the core logic so the notebook is self-contained
    print("agents/baseline.py not found — using inline baseline.")

    SUN_X, SUN_Y, SUN_SAFE_R = 50.0, 50.0, 12.0
    MAX_SPEED = 6.0
    MIN_GARRISON = 5

    def _dist(x1,y1,x2,y2): return math.sqrt((x1-x2)**2+(y1-y2)**2)
    def _speed(n): return 1.0 + (MAX_SPEED-1.0)*(math.log(max(n,1))/math.log(1000))**1.5
    def _sun_hit(x1,y1,x2,y2):
        dx,dy=x2-x1,y2-y1; fx,fy=x1-SUN_X,y1-SUN_Y
        a=dx*dx+dy*dy
        if a==0: return False
        b=2*(fx*dx+fy*dy); c=fx*fx+fy*fy-SUN_SAFE_R**2
        disc=b*b-4*a*c
        if disc<0: return False
        t1=(-b-math.sqrt(disc))/(2*a); t2=(-b+math.sqrt(disc))/(2*a)
        return (0<=t1<=1) or (0<=t2<=1) or (t1<0<t2)

    class _P:
        def __init__(self,id,owner,x,y,radius,ships,production):
            self.id=id;self.owner=owner;self.x=x;self.y=y
            self.radius=radius;self.ships=ships;self.production=production
    class _F:
        def __init__(self,id,owner,x,y,angle,from_planet_id,ships):
            self.id=id;self.owner=owner;self.x=x;self.y=y
            self.angle=angle;self.from_planet_id=from_planet_id;self.ships=ships

    def baseline_agent(obs, config=None):
        player = obs.player if hasattr(obs,'player') else obs.get('player',0)
        raw_p  = obs.planets if hasattr(obs,'planets') else obs.get('planets',[])
        raw_f  = obs.fleets  if hasattr(obs,'fleets')  else obs.get('fleets', [])
        planets=[_P(*p) for p in raw_p]; fleets=[_F(*f) for f in raw_f]
        mine=[p for p in planets if p.owner==player]
        targets=[p for p in planets if p.owner!=player]
        if not mine or not targets: return []
        moves=[]; claimed=set()
        for src in sorted(mine,key=lambda p:-p.production):
            avail=src.ships-MIN_GARRISON
            if avail<2: continue
            cands=[t for t in targets if t.id not in claimed
                   and not _sun_hit(src.x,src.y,t.x,t.y)]
            if not cands: continue
            best=min(cands, key=lambda t: _dist(src.x,src.y,t.x,t.y)+t.ships*0.5-t.production*10)
            d=_dist(src.x,src.y,best.x,best.y)
            tt=d/_speed(avail)
            need=int(best.ships+(best.production*tt if best.owner>=0 else 0))+1
            send=min(need,avail)
            if send<2: continue
            moves.append([src.id, math.atan2(best.y-src.y,best.x-src.x), send])
            claimed.add(best.id)
        return moves

In [ ]:
# Quick sanity check: baseline vs random (2-player)
env = make("orbit_wars", debug=False)
env.run([baseline_agent, "random"])
final = env.steps[-1]
for i, s in enumerate(final):
    print(f"Player {i}: reward={s.reward}, status={s.status}")
print(f"Total steps: {len(env.steps)}")

In [ ]:
# Benchmark: win-rate over N games
def win_rate(agent_a, agent_b, n_games=20, verbose=False):
    wins, losses, ties = 0, 0, 0
    for g in range(n_games):
        env = make("orbit_wars", debug=False)
        env.run([agent_a, agent_b])
        r = env.steps[-1][0].reward
        if r > 0: wins += 1
        elif r < 0: losses += 1
        else: ties += 1
        if verbose: print(f"  game {g+1}: {'W' if r>0 else 'L' if r<0 else 'T'}", end=' ')
    if verbose: print()
    return wins/n_games, losses/n_games, ties/n_games

w, l, t = win_rate(baseline_agent, "random", n_games=10)
print(f"Baseline vs random  —  W:{w:.0%}  L:{l:.0%}  T:{t:.0%}")

## 3. RL Training with PPO

We wrap the Kaggle environment in a Gymnasium interface, then train with
Stable-Baselines3 PPO.

**Action space:** `MultiDiscrete([MAX_OWN, MAX_PLANETS, N_FRACTIONS])`
→ one fleet launch per step: pick source planet slot, target planet slot, and 25/50/75/100% of ships.

**Observation:** fixed-size float32 vector (683 dims) encoding global state,
all planet features, and all fleet features (padded/truncated to max counts).

**Reward:** dense shaping (∆ships + ∆production) + terminal ±1.

In [ ]:
try:
    from orbit_wars_env import OrbitWarsEnv
    print("Imported OrbitWarsEnv from orbit_wars_env.py")
except ImportError:
    print("orbit_wars_env.py not found — using inline stub.")
    # Minimal stub so the notebook still runs; replace with full implementation.
    import gymnasium as gym
    from gymnasium import spaces

    MAX_PLANETS=40; MAX_FLEETS=80; MAX_OWN=12; N_FRAC=4
    OBS_DIM = 3 + MAX_PLANETS*7 + MAX_FLEETS*5  # 683
    SHIP_FRAC = [0.25, 0.50, 0.75, 1.00]

    class _SP:
        def __init__(self,id,owner,x,y,r,s,p):
            self.id=id;self.owner=owner;self.x=x;self.y=y;self.ships=s;self.production=p

    class OrbitWarsEnv(gym.Env):
        def __init__(self, opponent="random", reward_shaping=True, seed=None):
            super().__init__()
            self.opponent=opponent; self.reward_shaping=reward_shaping
            self.observation_space=spaces.Box(0.,1.,(OBS_DIM,),dtype=np.float32)
            self.action_space=spaces.MultiDiscrete([MAX_OWN, MAX_PLANETS, N_FRAC])
            self._env=None; self._step=1; self._done=False
            self._prev_ships=0; self._prev_prod=0

        def reset(self, seed=None, options=None):
            super().reset(seed=seed)
            cfg={"seed":seed} if seed else {}
            self._env=make("orbit_wars",configuration=cfg,debug=False)
            self._env.reset(); self._step=1; self._done=False
            obs0=self._env.steps[0][0].observation
            return self._encode(obs0), {}

        def step(self, action):
            if self._step>=len(self._env.steps):
                return np.zeros(OBS_DIM,np.float32),0.,True,False,{}
            own_s,tgt_s,frac=int(action[0]),int(action[1]),int(action[2])
            obs_raw=self._env.steps[self._step][0].observation
            move=self._decode(obs_raw,own_s,tgt_s,frac)
            opp_obs=self._env.steps[self._step][1].observation
            opp_move=self.opponent(opp_obs) if callable(self.opponent) else []
            try: self._env.step([move,opp_move])
            except: self._done=True; return self._encode(obs_raw),0.,True,False,{}
            self._step+=1
            nxt=self._env.steps[self._step][0].observation if self._step<len(self._env.steps) else self._env.steps[-1][0].observation
            status=self._env.steps[self._step-1][0].status
            term=status in("DONE","INVALID","TIMEOUT","ERROR")
            kaggle_r=float(self._env.steps[-1][0].reward or 0.)
            r=self._shape(nxt)+(kaggle_r if term else 0.)
            if term: self._done=True
            return self._encode(nxt),r,term,False,{}

        def _decode(self,obs_raw,own_s,tgt_s,frac):
            player=obs_raw.player if hasattr(obs_raw,'player') else obs_raw.get('player',0)
            raw_p=obs_raw.planets if hasattr(obs_raw,'planets') else obs_raw.get('planets',[])
            planets=[_SP(*p) for p in raw_p]
            mine=sorted([p for p in planets if p.owner==player],key=lambda p:p.id)
            if own_s>=len(mine) or tgt_s>=len(planets): return []
            src=mine[own_s]; tgt=planets[tgt_s]
            if tgt.id==src.id: return []
            ships=max(1,int(src.ships*SHIP_FRAC[frac]))
            if ships>=src.ships: ships=max(1,src.ships-1)
            return [[src.id,math.atan2(tgt.y-src.y,tgt.x-src.x),ships]]

        def _encode(self,obs_raw):
            v=np.zeros(OBS_DIM,np.float32)
            player=obs_raw.player if hasattr(obs_raw,'player') else obs_raw.get('player',0)
            raw_p=obs_raw.planets if hasattr(obs_raw,'planets') else obs_raw.get('planets',[])
            raw_f=obs_raw.fleets  if hasattr(obs_raw,'fleets')  else obs_raw.get('fleets',[])
            step=obs_raw.step if hasattr(obs_raw,'step') else obs_raw.get('step',0)
            av=obs_raw.angular_velocity if hasattr(obs_raw,'angular_velocity') else obs_raw.get('angular_velocity',0.)
            v[0]=step/500.; v[1]=av/0.05
            off=3
            for i,p in enumerate(raw_p[:MAX_PLANETS]):
                pl=_SP(*p); b=off+i*7
                v[b+0]=1. if pl.owner==player else 0.
                v[b+1]=1. if pl.owner>=0 and pl.owner!=player else 0.
                v[b+2]=1. if pl.owner==-1 else 0.
                v[b+3]=pl.x/100.; v[b+4]=pl.y/100.
                v[b+5]=math.log1p(pl.ships)/10.; v[b+6]=pl.production/5.
            off=3+MAX_PLANETS*7
            for i,f in enumerate(raw_f[:MAX_FLEETS]):
                fl=type('F',(),dict(zip(['id','owner','x','y','angle','from_planet_id','ships'],f)))()
                b=off+i*5
                v[b+0]=1. if fl.owner==player else 0.
                v[b+1]=1. if fl.owner!=player else 0.
                v[b+2]=fl.x/100.; v[b+3]=fl.y/100.; v[b+4]=math.log1p(fl.ships)/10.
            return v

        def _shape(self,obs_raw):
            player=obs_raw.player if hasattr(obs_raw,'player') else obs_raw.get('player',0)
            raw_p=obs_raw.planets if hasattr(obs_raw,'planets') else obs_raw.get('planets',[])
            raw_f=obs_raw.fleets  if hasattr(obs_raw,'fleets')  else obs_raw.get('fleets',[])
            planets=[_SP(*p) for p in raw_p]
            fl_ships=sum(f[-1] for f in raw_f if f[1]==player)
            ships=sum(p.ships for p in planets if p.owner==player)+fl_ships
            prod=sum(p.production for p in planets if p.owner==player)
            r=(ships-self._prev_ships)*0.001+(prod-self._prev_prod)*0.005
            self._prev_ships=ships; self._prev_prod=prod
            return r

In [ ]:
# Verify env
env_test = OrbitWarsEnv(opponent=baseline_agent)
obs, _ = env_test.reset(seed=42)
print(f"obs shape : {obs.shape}")
print(f"obs range : [{obs.min():.3f}, {obs.max():.3f}]")
print(f"action    : {env_test.action_space}")
action = env_test.action_space.sample()
obs2, r, term, trunc, _ = env_test.step(action)
print(f"step OK   : reward={r:.4f}, terminated={term}")

### 3b. PPO Training

Adjust `TOTAL_TIMESTEPS` based on available compute:
- Local CPU : 50 000 (smoke test)
- Kaggle CPU : 200 000
- Colab GPU  : 1 000 000+

In [ ]:
from stable_baselines3 import PPO
from stable_baselines3.common.env_util import make_vec_env
from stable_baselines3.common.callbacks import EvalCallback, CheckpointCallback
from stable_baselines3.common.monitor import Monitor
import os

# ── hyperparameters ────────────────────────────────────────────────────────
TOTAL_TIMESTEPS = 200_000   # increase for real training
N_ENVS = 4                  # parallel envs (reduce if memory-constrained)
MODEL_DIR = "models"
LOG_DIR   = "logs"
os.makedirs(MODEL_DIR, exist_ok=True)
os.makedirs(LOG_DIR,   exist_ok=True)

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Training on: {device}")

# ── vectorised training environment ───────────────────────────────────────
def make_train_env():
    e = OrbitWarsEnv(opponent=baseline_agent, reward_shaping=True)
    return Monitor(e)

vec_env = make_vec_env(make_train_env, n_envs=N_ENVS)

# ── model ──────────────────────────────────────────────────────────────────
model = PPO(
    "MlpPolicy",
    vec_env,
    learning_rate=3e-4,
    n_steps=512,
    batch_size=256,
    n_epochs=10,
    gamma=0.995,           # long episodes need high discount
    gae_lambda=0.95,
    clip_range=0.2,
    ent_coef=0.01,         # encourage exploration
    vf_coef=0.5,
    max_grad_norm=0.5,
    verbose=1,
    device=device,
    tensorboard_log=LOG_DIR,
)
print(model.policy)

In [ ]:
# ── callbacks ──────────────────────────────────────────────────────────────
eval_env = Monitor(OrbitWarsEnv(opponent=baseline_agent, reward_shaping=False))

eval_cb = EvalCallback(
    eval_env,
    best_model_save_path=MODEL_DIR,
    log_path=LOG_DIR,
    eval_freq=max(5_000 // N_ENVS, 1),
    n_eval_episodes=10,
    deterministic=True,
    verbose=1,
)
ckpt_cb = CheckpointCallback(
    save_freq=max(20_000 // N_ENVS, 1),
    save_path=MODEL_DIR,
    name_prefix="ppo_orbit",
)

# ── train ──────────────────────────────────────────────────────────────────
t0 = time.time()
model.learn(
    total_timesteps=TOTAL_TIMESTEPS,
    callback=[eval_cb, ckpt_cb],
    progress_bar=True,
)
print(f"Training finished in {time.time()-t0:.1f}s")
model.save(f"{MODEL_DIR}/ppo_orbit_final")
print(f"Saved to {MODEL_DIR}/ppo_orbit_final.zip")

## 4. Evaluation

Compare the trained RL agent against the deterministic baseline.

In [ ]:
# Load best model (saved by EvalCallback as best_model.zip)
best_model_path = f"{MODEL_DIR}/best_model"
if os.path.exists(best_model_path + ".zip"):
    model = PPO.load(best_model_path, device="cpu")
    print(f"Loaded {best_model_path}.zip")
else:
    print(f"best_model.zip not found, using last model.")

In [ ]:
def make_rl_agent(loaded_model):
    """Wrap a trained SB3 model as a Kaggle-compatible agent function."""
    _env_ref = OrbitWarsEnv()   # used only for obs encoding / action decoding

    def rl_agent(obs, config=None):
        obs_vec = _env_ref._obs_to_vec(obs, player=obs.player if hasattr(obs,'player') else obs.get('player',0))
        action, _ = loaded_model.predict(obs_vec[np.newaxis], deterministic=True)
        action = action[0]
        own_s, tgt_s, frac = int(action[0]), int(action[1]), int(action[2])
        return _env_ref._decode_action(obs, own_s, tgt_s, frac)

    return rl_agent

rl_agent = make_rl_agent(model)

In [ ]:
# Head-to-head: RL vs baseline
print("Evaluating RL agent vs baseline (10 games each side)...")
w1,l1,t1 = win_rate(rl_agent,       baseline_agent, n_games=10)
w2,l2,t2 = win_rate(baseline_agent, rl_agent,       n_games=10)
print(f"RL       vs baseline  →  W:{w1:.0%}  L:{l1:.0%}  T:{t1:.0%}")
print(f"Baseline vs RL        →  W:{w2:.0%}  L:{l2:.0%}  T:{t2:.0%}")
print(f"RL       vs random    →", end=' ')
w3,l3,t3 = win_rate(rl_agent, "random", n_games=10)
print(f"W:{w3:.0%}  L:{l3:.0%}  T:{t3:.0%}")

In [ ]:
# Watch one game
env_watch = make("orbit_wars", debug=True)
env_watch.run([rl_agent, baseline_agent])
final = env_watch.steps[-1]
print(f"RL reward: {final[0].reward}, Baseline reward: {final[1].reward}")
env_watch.render(mode="ipython", width=800, height=600)

## 5. Kaggle Submission

Two submission modes:

**A. Baseline only** (no model weights) — submit `submission.py` directly.
**B. RL agent** — model weights are base64-encoded and embedded in `submission.py`.

Run the cell matching the agent you want to submit.

In [ ]:
SUBMISSION_FILE = "submission.py"

# ── Option A: submit the deterministic baseline ────────────────────────────
baseline_submission = '''
import math

SUN_X, SUN_Y, SUN_SAFE_R = 50.0, 50.0, 12.0
MAX_SPEED = 6.0
MIN_GARRISON = 5

def _dist(x1,y1,x2,y2): return math.sqrt((x1-x2)**2+(y1-y2)**2)
def _speed(n): return 1.0+(MAX_SPEED-1.0)*(math.log(max(n,1))/math.log(1000))**1.5
def _sun(x1,y1,x2,y2):
    dx,dy=x2-x1,y2-y1; fx,fy=x1-SUN_X,y1-SUN_Y
    a=dx*dx+dy*dy
    if a==0: return False
    b=2*(fx*dx+fy*dy); c=fx*fx+fy*fy-SUN_SAFE_R**2
    d=b*b-4*a*c
    if d<0: return False
    t1=(-b-math.sqrt(d))/(2*a); t2=(-b+math.sqrt(d))/(2*a)
    return (0<=t1<=1)or(0<=t2<=1)or(t1<0<t2)

class _P:
    def __init__(self,id,owner,x,y,r,s,p):
        self.id=id;self.owner=owner;self.x=x;self.y=y;self.ships=s;self.production=p
class _F:
    def __init__(self,id,owner,x,y,angle,fp,ships):
        self.owner=owner;self.x=x;self.y=y;self.ships=ships

def agent(obs, config=None):
    player=obs.player if hasattr(obs,"player") else obs.get("player",0)
    raw_p=obs.planets if hasattr(obs,"planets") else obs.get("planets",[])
    raw_f=obs.fleets  if hasattr(obs,"fleets")  else obs.get("fleets",[])
    planets=[_P(*p) for p in raw_p]; fleets=[_F(*f) for f in raw_f]
    mine=[p for p in planets if p.owner==player]
    targets=[p for p in planets if p.owner!=player]
    if not mine or not targets: return []
    moves=[]; claimed=set()
    for src in sorted(mine,key=lambda p:-p.production):
        avail=src.ships-MIN_GARRISON
        if avail<2: continue
        cands=[t for t in targets if t.id not in claimed and not _sun(src.x,src.y,t.x,t.y)]
        if not cands: continue
        best=min(cands,key=lambda t:_dist(src.x,src.y,t.x,t.y)+t.ships*0.5-t.production*10)
        d=_dist(src.x,src.y,best.x,best.y)
        tt=d/_speed(avail)
        need=int(best.ships+(best.production*tt if best.owner>=0 else 0))+1
        send=min(need,avail)
        if send<2: continue
        moves.append([src.id,math.atan2(best.y-src.y,best.x-src.x),send])
        claimed.add(best.id)
    return moves
'''

with open(SUBMISSION_FILE, "w") as f:
    f.write(baseline_submission.strip())
print(f"Wrote baseline submission → {SUBMISSION_FILE}")

In [ ]:
# ── Option B: submit the trained RL agent ──────────────────────────────────
# Embeds model weights as base64 so the single-file submission is self-contained.

import base64, zipfile, io, tempfile

best_path = f"{MODEL_DIR}/best_model.zip"
if not os.path.exists(best_path):
    best_path = f"{MODEL_DIR}/ppo_orbit_final.zip"

if os.path.exists(best_path):
    with open(best_path, "rb") as f:
        weights_b64 = base64.b64encode(f.read()).decode()

    obs_dim = 683   # must match orbit_wars_env.OBS_DIM
    rl_submission = f'''
import math, os, base64, tempfile
import numpy as np

# ── embedded model weights ────────────────────────────────────────────────
_WEIGHTS_B64 = '{weights_b64}'

_MODEL = None

def _load_model():
    global _MODEL
    if _MODEL is not None:
        return _MODEL
    try:
        from stable_baselines3 import PPO
    except ImportError:
        return None
    with tempfile.NamedTemporaryFile(suffix=".zip", delete=False) as tmp:
        tmp.write(base64.b64decode(_WEIGHTS_B64))
        tmp_path = tmp.name
    _MODEL = PPO.load(tmp_path, device="cpu")
    os.unlink(tmp_path)
    return _MODEL

# ── observation encoder (mirrors orbit_wars_env._obs_to_vec) ──────────────
_MAX_P, _MAX_F = 40, 80
_OBS_DIM = 3 + _MAX_P * 7 + _MAX_F * 5  # {obs_dim}
_FRAC = [0.25, 0.50, 0.75, 1.00]

def _encode(obs_raw):
    v = np.zeros(_OBS_DIM, np.float32)
    player = obs_raw.player if hasattr(obs_raw,"player") else obs_raw.get("player",0)
    raw_p = obs_raw.planets if hasattr(obs_raw,"planets") else obs_raw.get("planets",[])
    raw_f = obs_raw.fleets  if hasattr(obs_raw,"fleets")  else obs_raw.get("fleets",[])
    step  = obs_raw.step if hasattr(obs_raw,"step") else obs_raw.get("step",0)
    av    = obs_raw.angular_velocity if hasattr(obs_raw,"angular_velocity") else obs_raw.get("angular_velocity",0.)
    v[0]=step/500.; v[1]=av/0.05
    off=3
    for i,p in enumerate(raw_p[:_MAX_P]):
        b=off+i*7
        v[b+0]=1. if p[1]==player else 0.
        v[b+1]=1. if p[1]>=0 and p[1]!=player else 0.
        v[b+2]=1. if p[1]==-1 else 0.
        v[b+3]=p[2]/100.; v[b+4]=p[3]/100.
        v[b+5]=math.log1p(p[5])/10.; v[b+6]=p[6]/5.
    off=3+_MAX_P*7
    for i,f in enumerate(raw_f[:_MAX_F]):
        b=off+i*5
        v[b+0]=1. if f[1]==player else 0.
        v[b+1]=1. if f[1]!=player else 0.
        v[b+2]=f[2]/100.; v[b+3]=f[3]/100.; v[b+4]=math.log1p(f[6])/10.
    return v

def _decode(obs_raw, own_s, tgt_s, frac):
    player = obs_raw.player if hasattr(obs_raw,"player") else obs_raw.get("player",0)
    raw_p  = obs_raw.planets if hasattr(obs_raw,"planets") else obs_raw.get("planets",[])
    class _P:
        def __init__(self,id,owner,x,y,r,s,p):
            self.id=id;self.owner=owner;self.x=x;self.y=y;self.ships=s
    planets=[_P(*p) for p in raw_p]
    mine=sorted([p for p in planets if p.owner==player],key=lambda p:p.id)
    if own_s>=len(mine) or tgt_s>=len(planets): return []
    src=mine[own_s]; tgt=planets[tgt_s]
    if tgt.id==src.id: return []
    ships=max(1,int(src.ships*_FRAC[frac]))
    if ships>=src.ships: ships=max(1,src.ships-1)
    return [[src.id, math.atan2(
        raw_p[tgt_s][3]-raw_p[tgt_s-len(planets)][3] if False else (tgt.y if hasattr(tgt,"y") else raw_p[tgt_s][3])
        - (src.y if hasattr(src,"y") else raw_p[own_s][3]),
        (tgt.x if hasattr(tgt,"x") else raw_p[tgt_s][2])
        - (src.x if hasattr(src,"x") else raw_p[own_s][2])
    ), ships]]

def agent(obs, config=None):
    m = _load_model()
    if m is None:
        return []
    obs_vec = _encode(obs)
    action, _ = m.predict(obs_vec[np.newaxis], deterministic=True)
    own_s, tgt_s, frac = int(action[0][0]), int(action[0][1]), int(action[0][2])
    player = obs.player if hasattr(obs,"player") else obs.get("player",0)
    raw_p  = obs.planets if hasattr(obs,"planets") else obs.get("planets",[])
    class _P2:
        def __init__(self,id,owner,x,y,r,s,p):
            self.id=id;self.owner=owner;self.x=x;self.y=y;self.ships=s
    planets=[_P2(*p) for p in raw_p]
    mine=sorted([p for p in planets if p.owner==player],key=lambda p:p.id)
    if own_s>=len(mine) or tgt_s>=len(planets): return []
    src=mine[own_s]; tgt=planets[tgt_s]
    if tgt.id==src.id: return []
    ships=max(1,int(src.ships*_FRAC[frac]))
    if ships>=src.ships: ships=max(1,src.ships-1)
    return [[src.id, math.atan2(tgt.y-src.y, tgt.x-src.x), ships]]
'''

    rl_sub_file = "submission_rl.py"
    with open(rl_sub_file, "w") as f:
        f.write(rl_submission.strip())
    print(f"Wrote RL submission → {rl_sub_file}  ({os.path.getsize(rl_sub_file)//1024} KB)")
    print("To submit: upload submission_rl.py to Kaggle and click Submit.")
else:
    print("No trained model found — train first (Section 3b), then re-run this cell.")

### Submitting to Kaggle

1. In the Kaggle competition page → **Submit** tab
2. Upload `submission.py` (baseline) or `submission_rl.py` (trained RL agent)
3. Kaggle evaluates your agent in live matches vs other participants

### Tips for improving the RL agent

| Technique | Why |
|-----------|-----|
| **Self-play** | Opponent improves alongside your agent; prevents overfitting to baseline |
| **Larger network** | `policy_kwargs=dict(net_arch=[256,256])` for more capacity |
| **Longer training** | 1M+ steps on GPU typically needed for meaningful results |
| **Curriculum** | Start vs random, then baseline, then self-play |
| **Action masking** | Mask invalid actions (MaskablePPO from sb3-contrib) |
| **Reward tuning** | Experiment with `reward_shaping` weights in `OrbitWarsEnv` |